# Landfill Dependency, Circular-Economy Regimes, and Big Data Analytics
### Reproducible analysis notebook — Panel OLS + Machine-Learning forecasting

This notebook reproduces the empirical analysis reported in the manuscript
*"Structural Determinants of Landfill Dependency in Emerging and Transition European
Economies: A Panel Analysis of Circular-Economy Regimes and the Mediating Role of
Digitalization."*

It is organized in two self-contained parts:

- **Part A — Panel OLS analysis** (9 countries, 2015–2023): descriptive statistics,
  CE–BDA regime profiles, regime validation (Welch ANOVA, Games–Howell), cluster
  stability (Adjusted Rand Index), four nested panel-OLS specifications with
  country-clustered (cluster-robust) standard errors, diagnostics, VIF, and the
  Albania profile. *Reproduces Tables 1–12.*
- **Part B — ML forecasting** (Municipality of Tirana, monthly 2017–2024): Naive,
  Seasonal-Naive and SARIMA benchmarks, an XGBoost model, a recursive business-as-usual
  (BAU) forecast to 2030, and policy-reduction scenarios. *Reproduces Tables 13–14.*

**Data.** Two input files are required (see the `config` cell):
`panel_landfill_2015_2023.csv` (country–year panel, with the pre-computed CE–BDA
cluster id) and `tirana_landfill_monthly_2017_2024.csv` (monthly landfilled tonnage).
Raw indicators derive from Eurostat, the World Bank (WDI) and the ITU.

**Reproducibility.** All stochastic steps use `RANDOM_STATE = 42`.


## 0. Setup

In [ ]:
# 0.1 — Install dependencies (Colab)
!pip -q install pingouin xgboost statsmodels scikit-learn
print("Dependencies installed.")

In [ ]:
# 0.2 — Imports
import os, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, mean_absolute_error, mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.graphics.gofplots import qqplot
from statsmodels.tsa.statespace.sarimax import SARIMAX

import pingouin as pg
from xgboost import XGBRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
print("Imports OK.")

In [ ]:
# 0.3 — Configuration
RANDOM_STATE = 42
OUTPUT_DIR   = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Input files. Set BASE_URL to your GitHub raw path to load directly, e.g.
#   BASE_URL = "https://raw.githubusercontent.com/<user>/<repo>/main/data/"
# Leave BASE_URL = "" to load local files (or use the Colab upload fallback below).
BASE_URL    = ""
PANEL_CSV   = "panel_landfill_2015_2023.csv"
MONTHLY_CSV = "tirana_landfill_monthly_2017_2024.csv"

def load_csv(name, **kwargs):
    """Load a CSV from BASE_URL if set, else from the local working directory;
    if not found locally, fall back to a Colab upload prompt."""
    src = (BASE_URL + name) if BASE_URL else name
    try:
        return pd.read_csv(src, **kwargs)
    except (FileNotFoundError, OSError):
        try:
            from google.colab import files
            print(f"'{name}' not found. Please upload it now:")
            up = files.upload()
            key = name if name in up else list(up.keys())[0]
            return pd.read_csv(key, **kwargs)
        except Exception as e:
            raise FileNotFoundError(
                f"Could not load '{name}'. Set BASE_URL or place the file next to the notebook."
            ) from e

print("Config ready. RANDOM_STATE =", RANDOM_STATE)

---
# Part A — Panel OLS Analysis (9 countries, 2015–2023)

Reproduces Tables 1–12. The CE–BDA cluster id is read from the panel file; it was
obtained by K-means (k = 3) on the standardized indicator vector in a prior data-preparation
step, and its stability is verified below via the Adjusted Rand Index (Section A.6).

### A.1 — Load and prepare the panel

In [ ]:
# Column map: original (Albanian) -> English working names. Unmatched columns are kept.
COLMAP = {
    "Shteti": "country", "Viti": "year",
    "norma_e_depozitimit_ne_landfill": "landfill_rate", "norma_landfill": "landfill_rate",
    "norma_e_trajtimit_jo_ne_landfill": "nonlandfill_rate", "norma_jo_landfill": "nonlandfill_rate",
    "mbetje_per_fryme": "waste_per_capita",
    "intensiteti_trajtimit": "treatment_intensity",
    "PBB_per_fryme": "gdp_per_capita",
    "popullesia_urbane_perqindja_mbi_popullesine_totale": "urban_pop", "popullesia_urbane": "urban_pop",
    "papunesia_perqindja_totale_fuqise_punetore": "unemployment", "papunesia": "unemployment",
    "perdorimi_internetit_nga_individe": "internet", "interneti": "internet",
    "cluster": "cluster_id",
}

panel = load_csv(PANEL_CSV)
panel = panel.rename(columns=COLMAP)
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)

# Time trend and regime dummies (reference = cluster 0)
panel["time_trend"] = panel["year"] - panel["year"].min()
dummies = pd.get_dummies(panel["cluster_id"], prefix="cluster", drop_first=True)
panel = pd.concat([panel, dummies], axis=1)
for c in ["cluster_1", "cluster_2"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce").fillna(0).astype(int)

REGIME_LABEL = {0: "Performers", 1: "High landfill / low GDP", 2: "High landfill / mid GDP"}
panel["regime"] = panel["cluster_id"].map(REGIME_LABEL)

analysis_cols = ["country", "year", "landfill_rate", "waste_per_capita",
                 "treatment_intensity", "gdp_per_capita", "urban_pop",
                 "unemployment", "internet", "time_trend", "cluster_id",
                 "cluster_1", "cluster_2"]
df_model = panel[[c for c in analysis_cols if c in panel.columns]].dropna().reset_index(drop=True)

print("Panel shape:", df_model.shape, "| countries:", df_model['country'].nunique(),
      "| years:", df_model['year'].min(), "-", df_model['year'].max())
df_model.head()

### A.2 — Descriptive statistics (Table 3)

In [ ]:
desc_vars = ["landfill_rate", "waste_per_capita", "treatment_intensity",
             "gdp_per_capita", "internet", "urban_pop", "unemployment"]
labels = {"landfill_rate": "Landfill rate (0-1)", "waste_per_capita": "Waste per capita (kg/inh.)",
          "treatment_intensity": "Treatment intensity (ratio)", "gdp_per_capita": "GDP per capita (index)",
          "internet": "Internet use (%)", "urban_pop": "Urban population (%)", "unemployment": "Unemployment (%)"}

desc = pd.DataFrame({
    "Variable": [labels[v] for v in desc_vars],
    "N":      [df_model[v].notna().sum() for v in desc_vars],
    "Mean":   [df_model[v].mean() for v in desc_vars],
    "SD":     [df_model[v].std() for v in desc_vars],
    "Min":    [df_model[v].min() for v in desc_vars],
    "Max":    [df_model[v].max() for v in desc_vars],
    "Median": [df_model[v].median() for v in desc_vars],
}).round(3)
desc.to_csv(f"{OUTPUT_DIR}/Table03_descriptives.csv", index=False)
desc

### A.3 — CE–BDA regime profiles and classification (Tables 4–5)

In [ ]:
# Table 5 — mean structural profile by regime
profile_vars = ["waste_per_capita", "landfill_rate", "nonlandfill_rate", "treatment_intensity",
                "gdp_per_capita", "unemployment", "urban_pop", "internet"]
profile_vars = [v for v in profile_vars if v in panel.columns]
profiles = (panel.groupby(["cluster_id", "regime"])[profile_vars]
            .mean().round(3).reset_index())
profiles.to_csv(f"{OUTPUT_DIR}/Table05_regime_profiles.csv", index=False)
profiles

In [ ]:
# Table 4 — dominant regime and share of years per country
def classify(g):
    counts = g["cluster_id"].value_counts()
    dom = counts.idxmax()
    return pd.Series({
        "dominant_regime": f"Cl. {dom}",
        "yrs_Cl0": int((g["cluster_id"] == 0).sum()),
        "yrs_Cl1": int((g["cluster_id"] == 1).sum()),
        "yrs_Cl2": int((g["cluster_id"] == 2).sum()),
        "total_yrs": len(g),
        "pct_dominant": round(100 * counts.max() / len(g), 1),
    })
classification = panel.groupby("country").apply(classify).reset_index()
classification.to_csv(f"{OUTPUT_DIR}/Table04_classification.csv", index=False)
classification

### A.4 — Regime validation: Welch ANOVA and Games–Howell (Tables 6–7)

In [ ]:
anova_vars = ["landfill_rate", "waste_per_capita", "gdp_per_capita",
              "urban_pop", "treatment_intensity", "unemployment", "internet"]

welch_rows = []
for v in anova_vars:
    tmp = df_model[[v, "cluster_id"]].dropna()
    r = pg.welch_anova(data=tmp, dv=v, between="cluster_id").iloc[0]
    welch_rows.append({"Indicator": v, "Welch_F": round(r["F"], 2),
                       "df_num": r["ddof1"], "df_den": round(r["ddof2"], 1),
                       "p_value": r["p-unc"]})
welch = pd.DataFrame(welch_rows)
welch.to_csv(f"{OUTPUT_DIR}/Table06_welch_anova.csv", index=False)
welch

In [ ]:
gh_frames = []
for v in anova_vars:
    tmp = df_model[[v, "cluster_id"]].dropna()
    gh = pg.pairwise_gameshowell(data=tmp, dv=v, between="cluster_id")
    gh["Indicator"] = v
    gh_frames.append(gh)
games_howell = pd.concat(gh_frames, ignore_index=True)

pcol = "pval" if "pval" in games_howell.columns else ("p-tukey" if "p-tukey" in games_howell.columns else "p-unc")
keep = [c for c in ["Indicator", "A", "B", "mean(A)", "mean(B)", "diff", "se", "T", "df", pcol, "hedges"]
        if c in games_howell.columns]
games_howell = games_howell[keep].round(4)
games_howell.to_csv(f"{OUTPUT_DIR}/Table07_games_howell.csv", index=False)
games_howell

### A.5 — Landfill trend by regime (Table 8)

In [ ]:
trend_rows = []
for cl, g in df_model.groupby("cluster_id"):
    g2 = g[["year", "landfill_rate"]].dropna()
    m = sm.OLS(g2["landfill_rate"], sm.add_constant(g2["year"])).fit()
    trend_rows.append({"Regime": f"Cl. {cl}", "Typology": REGIME_LABEL.get(cl, ""),
                       "Slope_beta": round(m.params["year"], 4),
                       "p_value": round(m.pvalues["year"], 4), "R2": round(m.rsquared, 4)})
trend = pd.DataFrame(trend_rows)
trend.to_csv(f"{OUTPUT_DIR}/Table08_trend_by_regime.csv", index=False)
trend

### A.6 — Cluster stability across scaling strategies: ARI (Table 9)

In [ ]:
cluster_features = ["waste_per_capita", "landfill_rate", "treatment_intensity",
                    "gdp_per_capita", "unemployment", "urban_pop", "internet"]
cluster_features = [c for c in cluster_features if c in df_model.columns]
Xraw = df_model[cluster_features].dropna()

X_std      = StandardScaler().fit_transform(Xraw)
X_std_clip = np.clip(X_std, -3, 3)
X_rob      = RobustScaler().fit_transform(Xraw)

km = lambda X: KMeans(n_clusters=3, n_init=20, random_state=RANDOM_STATE).fit(X).labels_
a, b, c = km(X_std), km(X_std_clip), km(X_rob)

ari = pd.DataFrame(
    [[1.0, adjusted_rand_score(a, b), adjusted_rand_score(a, c)],
     [adjusted_rand_score(b, a), 1.0, adjusted_rand_score(b, c)],
     [adjusted_rand_score(c, a), adjusted_rand_score(c, b), 1.0]],
    index=["Standard", "Standard+Clip", "Robust"],
    columns=["Standard", "Standard+Clip", "Robust"]).round(3)
ari.to_csv(f"{OUTPUT_DIR}/Table09_ARI.csv")
ari

### A.7 — Panel OLS: four nested specifications (Table 10)

All models are estimated by OLS with **country-clustered (cluster-robust) standard errors**.
The dependent variable is the landfill rate (0–1).

- **M_VAR** — economic and digital variables only (no regime dummies)
- **M_KL** — regime dummies only
- **M1** — variables + regimes, without digitalization
- **M2** — full model (adds internet use, the BDA proxy)

In [ ]:
YVAR = "landfill_rate"
VARS_VAR_ONLY = ["waste_per_capita", "treatment_intensity", "gdp_per_capita",
                 "urban_pop", "unemployment", "internet", "time_trend"]
VARS_KL_ONLY  = ["cluster_1", "cluster_2"]
VARS_M1 = ["waste_per_capita", "treatment_intensity", "gdp_per_capita",
           "urban_pop", "unemployment", "time_trend", "cluster_1", "cluster_2"]
VARS_M2 = ["waste_per_capita", "treatment_intensity", "gdp_per_capita",
           "urban_pop", "unemployment", "internet", "time_trend", "cluster_1", "cluster_2"]

def build_X(df_in, var_list):
    d = df_in[[YVAR, "country"] + var_list].copy()
    for c in [YVAR] + var_list:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    d = d.dropna(subset=[YVAR] + var_list)
    X = sm.add_constant(d[var_list].astype(float), has_constant="add")
    return d, d[YVAR].astype(float), X, d["country"].astype(str)

def fit_ols_cluster(df_in, var_list):
    d, y, X, groups = build_X(df_in, var_list)
    return sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": groups}), d, y, X, groups

def coef_table(m, name):
    return pd.DataFrame({"Variable": m.params.index,
                         f"{name}_beta": m.params.values.round(4),
                         f"{name}_p":    m.pvalues.values.round(4)})

ols_mvar, *_ = fit_ols_cluster(df_model, VARS_VAR_ONLY)
ols_mkl , *_ = fit_ols_cluster(df_model, VARS_KL_ONLY)
ols_m1  , _, _, X_m1, _ = fit_ols_cluster(df_model, VARS_M1)
ols_m2  , _, _, X_m2, _ = fit_ols_cluster(df_model, VARS_M2)

coef = (coef_table(ols_mvar, "M_VAR")
        .merge(coef_table(ols_mkl, "M_KL"), on="Variable", how="outer")
        .merge(coef_table(ols_m1,  "M1"),   on="Variable", how="outer")
        .merge(coef_table(ols_m2,  "M2"),   on="Variable", how="outer"))

stats_tbl = pd.DataFrame({
    "Model": ["M_VAR", "M_KL", "M1", "M2"],
    "R2":     [round(m.rsquared, 3)     for m in (ols_mvar, ols_mkl, ols_m1, ols_m2)],
    "Adj_R2": [round(m.rsquared_adj, 3) for m in (ols_mvar, ols_mkl, ols_m1, ols_m2)],
    "F":      [round(m.fvalue, 1)       for m in (ols_mvar, ols_mkl, ols_m1, ols_m2)],
    "N":      [int(m.nobs)              for m in (ols_mvar, ols_mkl, ols_m1, ols_m2)],
})
coef.to_csv(f"{OUTPUT_DIR}/Table10_OLS_coefficients.csv", index=False)
stats_tbl.to_csv(f"{OUTPUT_DIR}/Table10_OLS_modelstats.csv", index=False)
print("Model fit statistics:"); display(stats_tbl)
print("\nCoefficients (beta, p) by specification:"); display(coef)

In [ ]:
# Full M2 summary (cluster-robust)
print(ols_m2.summary())

### A.8 — OLS diagnostics: Durbin–Watson, Jarque–Bera, residual plots (Table 2)

In [ ]:
def durbin_watson(res):
    d = np.diff(res); return float(np.sum(d**2) / np.sum(res**2))

diag_rows = []
for m, name in [(ols_m1, "M1"), (ols_m2, "M2")]:
    jb, jb_p, skew, kurt = sm.stats.jarque_bera(m.resid)
    diag_rows.append({"Model": name, "Durbin_Watson": round(durbin_watson(m.resid), 3),
                      "Jarque_Bera": round(jb, 2), "JB_p": round(jb_p, 4),
                      "Skew": round(skew, 3), "Kurtosis": round(kurt, 3)})
diag = pd.DataFrame(diag_rows)
diag.to_csv(f"{OUTPUT_DIR}/Table02_diagnostics.csv", index=False)
display(diag)

fig = plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.scatter(ols_m2.fittedvalues, ols_m2.resid, s=18)
plt.axhline(0, ls="--"); plt.xlabel("Fitted"); plt.ylabel("Residuals")
plt.title("Residuals vs Fitted — M2"); plt.grid(alpha=.2)
plt.subplot(1, 2, 2)
qqplot(ols_m2.resid, line="45", fit=True, ax=plt.gca()); plt.title("Q–Q plot — M2")
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/Fig_diagnostics_M2.png", dpi=200, bbox_inches="tight"); plt.show()

### A.9 — Multicollinearity: VIF (Table 11)

In [ ]:
def vif_table(X):
    rows = [(c, variance_inflation_factor(X.values, i)) for i, c in enumerate(X.columns)]
    out = pd.DataFrame(rows, columns=["Variable", "VIF"])
    out = out[out["Variable"] != "const"].copy()
    out["Interpretation"] = np.where(out["VIF"] > 10, "High",
                              np.where(out["VIF"] > 5, "Moderate", "Acceptable"))
    return out.sort_values("VIF", ascending=False).round(2).reset_index(drop=True)

vif = vif_table(X_m2)
vif.to_csv(f"{OUTPUT_DIR}/Table11_VIF.csv", index=False)
vif

### A.10 — Albania relative to the European typologies (Table 12)

In [ ]:
prof_vars = ["landfill_rate", "waste_per_capita", "gdp_per_capita",
             "treatment_intensity", "urban_pop", "unemployment", "internet"]
prof_vars = [v for v in prof_vars if v in panel.columns]

alb = panel[panel["country"].str.contains("Alban", case=False, na=False)][prof_vars].mean()
by_regime = panel.groupby("cluster_id")[prof_vars].mean()

albania = pd.DataFrame({"Indicator": prof_vars, "Albania": alb.values.round(3)})
for cl in sorted(by_regime.index):
    albania[f"Cl.{cl}"] = by_regime.loc[cl, prof_vars].values.round(3)
albania.to_csv(f"{OUTPUT_DIR}/Table12_albania_profile.csv", index=False)
albania

---
# Part B — ML Forecasting (Municipality of Tirana, monthly 2017–2024)

Reproduces Tables 13–14. The monthly series of landfilled tonnage is used to benchmark
an XGBoost model against Naive, Seasonal-Naive and SARIMA models, to build a recursive
business-as-usual (BAU) forecast to 2030, and to quantify policy-reduction scenarios.

The feature engineering avoids look-ahead leakage: all moving averages are shifted so that
`roll_k` at month *t* uses months *t−2 … t−(k+1)*, identical in training and recursive
inference. `year_offset` is used instead of the raw year because tree models do not
extrapolate beyond the training range (documented as a limitation).

### B.1 — Load and prepare the monthly series

In [ ]:
MONTH_MAP = {"Janar":1,"Shkurt":2,"Mars":3,"Prill":4,"Maj":5,"Qershor":6,
             "Korrik":7,"Gusht":8,"Shtator":9,"Tetor":10,"Nentor":11,"Dhjetor":12}
MONTH_RE = re.compile(r"^(%s)\s+(\d{2})$" % "|".join(MONTH_MAP))
TRAIN_END, TEST_START, BASE_YEAR = "2023-12-01", "2024-01-01", 2017

raw = load_csv(MONTHLY_CSV, encoding="latin1")
raw = raw.loc[:, ~raw.columns.astype(str).str.contains("Unnamed|Sh.nim", case=False)]
raw.columns = [str(c).replace("Shatator", "Shtator") for c in raw.columns]
month_cols = [c for c in raw.columns if MONTH_RE.match(str(c).strip())]

long = raw.melt(id_vars=["Treguesi"], value_vars=month_cols,
                var_name="month_label", value_name="tons_raw")

def parse_label(s):
    m = MONTH_RE.match(str(s).strip())
    return pd.Timestamp(year=2000+int(m.group(2)), month=MONTH_MAP[m.group(1)], day=1) if m else pd.NaT
long["date"] = long["month_label"].apply(parse_label)

def to_float(x):
    if pd.isna(x): return np.nan
    s = str(x).strip().replace(" ", "")
    s = s.replace(",", "") if ("," in s and "." in s) else s.replace(",", ".")
    s = re.sub(r"[^0-9.\-]", "", s)
    try: return float(s)
    except: return np.nan
long["tons"] = long["tons_raw"].apply(to_float)

y = (long[long["Treguesi"].astype(str).str.strip().str.lower() == "gjithsej"]
     .dropna(subset=["date"]).set_index("date")["tons"].sort_index().asfreq("MS"))
n_missing = int(y.isna().sum())
y = y.interpolate(method="linear", limit_direction="both")
print(f"Series: {y.index.min().date()} -> {y.index.max().date()} | N={len(y)} | interpolated={n_missing}")
y.tail()

### B.2 — Benchmarks: Naive, Seasonal Naive, SARIMA

In [ ]:
def rmse(a, b): return float(np.sqrt(mean_squared_error(np.array(a), np.array(b))))
train, test = y[:TRAIN_END], y[TEST_START:]

naive_pred = pd.Series(float(train.iloc[-1]), index=test.index)
seasonal_naive = pd.Series({dt: y.loc[dt - pd.DateOffset(years=1)] for dt in test.index})

sarima = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12),
                 enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
sarima_pred = sarima.get_forecast(steps=len(test)).predicted_mean
sarima_pred.index = test.index

for nm, p in [("Naive", naive_pred), ("Seasonal Naive", seasonal_naive), ("SARIMA", sarima_pred)]:
    print(f"{nm:15s} MAE={mean_absolute_error(test, p):.2f}  RMSE={rmse(test, p):.2f}")

### B.3 — XGBoost: feature engineering (leakage-safe) and training

In [ ]:
def build_features(series):
    d = series.to_frame("y")
    d["month"] = d.index.month
    d["year_offset"] = d.index.year - BASE_YEAR
    for lag in [1, 2, 3, 6, 12]:
        d[f"lag_{lag}"] = d["y"].shift(lag)
    d["roll_3"]  = d["y"].shift(1).rolling(3).mean()    # mean(t-2, t-3, t-4)
    d["roll_12"] = d["y"].shift(1).rolling(12).mean()   # mean(t-2, ..., t-13)
    return d.dropna()

feat = build_features(y)
train_df, test_df = feat.loc[:TRAIN_END], feat.loc[TEST_START:]
val_size = max(6, int(len(train_df) * 0.15))
fit_df, val_df = train_df.iloc[:-val_size], train_df.iloc[-val_size:]
feature_cols = [c for c in feat.columns if c != "y"]

xgb = XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=5,
                   subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                   early_stopping_rounds=50, random_state=RANDOM_STATE)
xgb.fit(fit_df[feature_cols], fit_df["y"],
        eval_set=[(val_df[feature_cols], val_df["y"])], verbose=False)

xgb_pred = pd.Series(xgb.predict(test_df[feature_cols]), index=test_df.index)
print(f"XGBoost (best_iteration={xgb.best_iteration})  "
      f"MAE={mean_absolute_error(test_df['y'], xgb_pred):.2f}  RMSE={rmse(test_df['y'], xgb_pred):.2f}")

### B.4 — Model accuracy comparison (Table 13)

In [ ]:
acc = pd.DataFrame({
    "Model": ["Naive", "Seasonal Naive", "SARIMA(1,1,1)(1,1,1,12)", "XGBoost"],
    "MAE (t)":  [round(mean_absolute_error(test, naive_pred), 2),
                 round(mean_absolute_error(test, seasonal_naive), 2),
                 round(mean_absolute_error(test, sarima_pred), 2),
                 round(mean_absolute_error(test_df["y"], xgb_pred), 2)],
    "RMSE (t)": [round(rmse(test, naive_pred), 2), round(rmse(test, seasonal_naive), 2),
                 round(rmse(test, sarima_pred), 2), round(rmse(test_df["y"], xgb_pred), 2)],
})
acc.to_csv(f"{OUTPUT_DIR}/Table13_model_accuracy.csv", index=False)
print("Best RMSE:", acc.loc[acc['RMSE (t)'].idxmin(), 'Model'],
      "| Best MAE:", acc.loc[acc['MAE (t)'].idxmin(), 'Model'])
print("Note: RMSE is the primary criterion (penalizes large errors relevant for "
      "landfill-capacity planning). A lower Naive MAE reflects early-2024 months close "
      "to the Dec-2023 reference level, not genuine predictive superiority.")
acc

### B.5 — Recursive business-as-usual (BAU) forecast, 2025–2030

In [ ]:
def make_features(history, dt):
    row = {"month": dt.month, "year_offset": dt.year - BASE_YEAR,
           "lag_1": history.iloc[-1], "lag_2": history.iloc[-2], "lag_3": history.iloc[-3],
           "lag_6": history.iloc[-6], "lag_12": history.iloc[-12],
           "roll_3":  float(history.iloc[-4:-1].mean()),
           "roll_12": float(history.iloc[-13:-1].mean())}
    return pd.DataFrame([row], columns=feature_cols)

future = pd.date_range("2025-01-01", "2030-12-01", freq="MS")
history, bau_preds = y.copy(), []
for dt in future:
    yhat = max(float(xgb.predict(make_features(history, dt))[0]), 0.0)
    bau_preds.append(yhat); history.loc[dt] = yhat
bau = pd.Series(bau_preds, index=future)

bau_annual = bau.groupby(bau.index.year).sum().round(2)
bau_annual.index.name = "Year"
bau_annual.to_frame("BAU total (t)").to_csv(f"{OUTPUT_DIR}/Table_BAU_annual.csv")
print("Annual BAU totals (t):"); print(bau_annual.to_string())
print("\nNote: tree models do not extrapolate beyond the training range, so the long-run "
      "BAU path is a level projection, not a continued trend.")

### B.6 — Policy-reduction scenarios (Table 14)

In [ ]:
yrs = bau.index.year; y0 = int(yrs.min())
moderate   = bau * ((1 - 0.02) ** (yrs - y0))
aggressive = bau * ((1 - 0.04) ** (yrs - y0))

summary = pd.DataFrame({"BAU": bau, "Moderate": moderate, "Aggressive": aggressive})
totals = summary.sum().to_frame("Total 2025-2030 (t)")
totals["Reduction vs BAU (t)"] = (totals.loc["BAU", "Total 2025-2030 (t)"] - totals["Total 2025-2030 (t)"]).round(2)
totals["Reduction (%)"] = (100 * totals["Reduction vs BAU (t)"] / totals.loc["BAU", "Total 2025-2030 (t)"]).round(2)
totals = totals.round(2)
totals.to_csv(f"{OUTPUT_DIR}/Table14_scenarios.csv")
totals

### B.7 — Figures

In [ ]:
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

# Test-set fit
plt.figure(figsize=(11, 4))
plt.plot(test.index, test.values, "o-", label="Actual")
plt.plot(xgb_pred.index, xgb_pred.values, "s--", label="XGBoost")
plt.title("Actual vs. forecast — test set (2024)"); plt.ylabel("Tonnes"); plt.legend(); plt.grid(alpha=.2)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/Fig_test_fit.png", dpi=200, bbox_inches="tight"); plt.show()

# BAU + scenarios (annual)
annual = summary.groupby(summary.index.year).sum()
plt.figure(figsize=(11, 4))
for col, style in [("BAU", "o-"), ("Moderate", "s--"), ("Aggressive", "^--")]:
    plt.plot(annual.index, annual[col], style, label=col)
plt.title("Annual landfilled waste — BAU vs. policy scenarios (2025–2030)")
plt.ylabel("Tonnes/year"); plt.legend(); plt.grid(alpha=.2)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/Fig_scenarios.png", dpi=200, bbox_inches="tight"); plt.show()

---
## Reproducibility notes

- **Environment.** `pandas`, `numpy`, `scikit-learn`, `statsmodels`, `scipy`, `pingouin`,
  `xgboost`, `matplotlib`. Pin versions in `requirements.txt` for exact reproduction.
- **Seed.** `RANDOM_STATE = 42` for K-means and XGBoost.
- **Panel regimes.** The `cluster_id` in the panel file was produced by K-means (k = 3)
  on the standardized indicators; stability is verified by the ARI matrix (Section A.6).
  Re-deriving clusters in-notebook may permute cluster labels; the stored `cluster_id`
  is used so that the OLS regime dummies match the published tables.
- **Outputs.** All tables are written to `outputs/` as CSV (Table02–Table14).
- **Data sources.** Eurostat (`env_wasmun`, `nama_10_pc`, `une_rt_a`), World Bank WDI
  (urban population), ITU / World Bank (internet use); Municipality of Tirana (monthly
  landfilled tonnage). See `data/DATA_SOURCES.md`.
